In [1]:
from multiprocessing import set_start_method
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# **Must** happen before torch or vllm ever touches CUDA
set_start_method("spawn", force=True)

from datasets import interleave_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizerFast, PreTrainedModel
from transformers import Trainer, TrainingArguments
from trl import SFTConfig, SFTTrainer
from trl import setup_chat_format
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset, Dataset
from concurrent.futures import ThreadPoolExecutor
from trl import DataCollatorForCompletionOnlyLM
import torch
from vllm import LLM, SamplingParams
from vllm.model_executor import set_random_seed as vllm_set_random_seed
from drgrpo_grader import r1_zero_reward_fn
import gc
from unittest.mock import patch
import wandb
import safetensors
import os
import json
import numpy as np
import time
import random 
import operator
from transformers import TrainerCallback, TrainerState, TrainerControl

model_name = "Qwen/Qwen2.5-1.5B"
tokenizer_dir =  "checkpoint/sft/tokenizer"
initial_sft_model_dir = "./checkpoint/sft/sft_lora_results3/my_merged_model"
training_data_output_dir = './dataset/expert_iteration1.hf'

tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir, local_files_only=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer.padding_side = "left"
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     ).to(device)



ds = load_dataset("openai/gsm8k", "main")

def preprocess_dataset(ds, usage, data_sampling_ratio=1, seed=125):
    sample_cnt = int(len(ds[usage]["answer"]) * data_sampling_ratio)
    random.seed(seed)
    sampled_data_idx = random.sample(range(0, len(ds[usage]["answer"])), sample_cnt)
    getter = operator.itemgetter(*sampled_data_idx)
    questions, answers =  getter(ds[usage]["question"]), getter(ds[usage]["answer"])
    print(len(questions), len(answers))
    with open("prompts/r1_zero.prompt", "r", encoding="utf-8") as f:
        prompt_string = f.read()

    def process_question(q):
        return prompt_string.format(question=q)
    def process_ground_truth(ans):
        return ans.split('\n#### ')[1]
    def process_prompt_completion(q, ans):
        prompt = prompt_string.format(question=q)
        cot =' ' + ans.split('\n#### ')[0] + ' </think>'
        gt = f" <answer> {ans.split('\n#### ')[1]} </answer>"
        return prompt + cot + gt
    with ThreadPoolExecutor() as executor:
        question_prompts = list(executor.map(process_question, ds[usage]["question"]))
    with ThreadPoolExecutor() as executor:
        ground_truth = list(executor.map(process_ground_truth, ds[usage]["answer"]))
    with ThreadPoolExecutor() as executor:
        prompt_completion = list(executor.map(process_prompt_completion, ds[usage]["question"], ds[usage]["answer"]))
    return question_prompts, ground_truth, prompt_completion

training_question_prompt, training_gt, training_data = preprocess_dataset(ds, 'train', 0.4)
test_prompt, test_gt =  preprocess_dataset(ds, 'test')[0], preprocess_dataset(ds, 'test')[1]
train_ds = Dataset.from_dict({
    "text":training_data,
    "prompt_temp":  training_question_prompt,
    "gt":training_gt
    })
val_ds = Dataset.from_dict({
    "prompt_temp": test_prompt[:len(test_prompt)//2],
    "gt": test_gt[:len(test_gt)//2],

})
test_ds = Dataset.from_dict({
    "prompt_temp": test_prompt[len(test_prompt)//2:],
    "gt": test_gt[len(test_gt)//2:],

})
# val_ds = Dataset.from_dict({
#     "prompt": test_prompt,
#     "gt": test_gt,

# })


# Build a collator whose response_template matches your prompt ending

collator = DataCollatorForCompletionOnlyLM(
    tokenizer = tokenizer,
    # Anything before *and including* this string gets label = -100
    response_template  = r"Assistant: <think>",   # note the space after >
    
)



INFO 06-26 08:16:46 [__init__.py:243] Automatically detected platform cuda.
2989 2989
1319 1319
1319 1319


In [2]:
len(tokenizer)

151666

In [2]:
def init_vllm(hf_model_dir: str,  device: str, seed: int, gpu_memory_utilization: float = 0.85):
    """Start the inference process, here we use vLLM to hold a model on
    a GPU separate from the policy.
    """
    vllm_set_random_seed(seed)
   
    # Monkeypatch from TRL:
    # https://github.com/huggingface/trl/blob/
    # 22759c820867c8659d00082ba8cf004e963873c1/trl/trainer/grpo_trainer.py
    # Patch vLLM to make sure we can
    # (1) place the vLLM model on the desired device (world_size_patch) and
    # (2) avoid a test that is not designed for our setting (profiling_patch).
    world_size_patch = patch("torch.distributed.get_world_size", return_value=1)
    profiling_patch = patch(
    "vllm.worker.worker.Worker._assert_memory_footprint_increased_during_profiling",
    return_value=None
    )
    with world_size_patch, profiling_patch:
        return LLM(
        #model="Qwen/Qwen2.5-1.5B",    # base model dir or HF name
        #peft_model="./checkpoint/sft/sft_lora_results3/checkpoint-1500",  # path to LoRA adapter
        model=hf_model_dir,
        #tokenizer=tokenizer,
        #tokenizer_mode='auto',
        device=device,
        dtype=torch.bfloat16,
        enable_prefix_caching=True,
        gpu_memory_utilization=gpu_memory_utilization,
        
        )

def data_filter(vllm, model_dir, ds, output_dir, G=8, sampling_temperature=1,top_p=1.0,sampling_max_tokens=1024,sampling_min_tokens=4, seed=233):
   
    ds_prompt = []
    ds_gt = []
    ds_prompt_completion = []
    not_valid = 0
    valid = 0

    
    start_time = time.time()
    sampling_params = SamplingParams(
        temperature=sampling_temperature,
        top_p=top_p,
        max_tokens=sampling_max_tokens,
        min_tokens=sampling_min_tokens,
        n=G,
        seed=seed,
        stop=["</answer>"]
    )
    sampling_params.include_stop_str_in_output = True
    all_prompt_texts = ds['prompt_temp']
    all_answer_gt = ds['gt']
    all_prompt_completion = ds['text']
    all_model_outputs = vllm.generate(all_prompt_texts, sampling_params)
    start2_time = time.time()
    print(f'generate all response time: {start2_time - start_time}------------')
    
    for i, (outputs, gt) in enumerate(zip(all_model_outputs, all_answer_gt)):
        prompt = outputs.prompt
        for output in outputs.outputs:
            
            generated_text = output.text
            res = r1_zero_reward_fn(generated_text, gt)
            if res['reward'] == 1:
                ds_prompt.append(prompt)
                ds_gt.append(gt)
                ds_prompt_completion.append(prompt+generated_text)
                valid +=1
            if res['reward'] ==0:
                not_valid +=1


    print(f'valid sample: {valid} ; not valid sample: {not_valid}')
    filtered_ds = Dataset.from_dict({
        "text":ds_prompt_completion,
        "prompt_temp":  ds_prompt,
        "gt":ds_gt
    })

    # for split, split_dataset in filtered_ds.items():
    #     output_dir= f"{output_dir.split('.jsonl')[0]}_{split}.jsonl"
    #     split_dataset.to_json(output_dir)

    filtered_ds.save_to_disk(output_dir)
    return filtered_ds




   


In [3]:

vllm = init_vllm(initial_sft_model_dir, device, 233)
training_question_prompt, training_gt, training_data = preprocess_dataset(ds, 'train', 0.4, seed=21)

train_ds = Dataset.from_dict({
    "text":training_data,
    "prompt_temp":  training_question_prompt,
    "gt":training_gt
    })

filtered_train_ds = data_filter(vllm, initial_sft_model_dir, train_ds, training_data_output_dir, G=4, sampling_temperature=0.4,top_p=1.0,sampling_max_tokens=1024,sampling_min_tokens=4, seed=233)

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     ).to(device)


INFO 06-26 05:38:09 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-26 05:38:09 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 06-26 05:38:09 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-26 05:38:18 [config.py:793] This model supports multiple tasks: {'generate', 'reward', 'embed', 'classify', 'score'}. Defaulting to 'generate'.
INFO 06-26 05:38:18 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-26 05:38:23 [__init__.py:243] Automatically detected platform cuda.
INFO 06-26 05:38:26 [core.py:438] Waiting for init message from front-end.
INFO 06-26 05:38:26 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-26 05:38:26 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.20it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.19it/s]



INFO 06-26 05:38:28 [default_loader.py:280] Loading weights took 0.56 seconds
INFO 06-26 05:38:28 [gpu_model_runner.py:1549] Model loading took 2.9094 GiB and 0.734087 seconds
INFO 06-26 05:38:35 [backends.py:459] Using cache directory: /home/sagemaker-user/.cache/vllm/torch_compile_cache/1871403a93/rank_0_0 for vLLM's torch.compile
INFO 06-26 05:38:35 [backends.py:469] Dynamo bytecode transform time: 6.86 s
INFO 06-26 05:38:40 [backends.py:132] Directly load the compiled graph(s) for shape None from the cache, took 5.092 s
INFO 06-26 05:38:41 [monitor.py:33] torch.compile takes 6.86 s in total
INFO 06-26 05:38:42 [kv_cache_utils.py:637] GPU KV cache size: 521,008 tokens
INFO 06-26 05:38:42 [kv_cache_utils.py:640] Maximum concurrency for 131,072 tokens per request: 3.97x
INFO 06-26 05:39:02 [gpu_model_runner.py:1933] Graph capturing finished in 20 secs, took 0.45 GiB
INFO 06-26 05:39:03 [core.py:167] init engine (profile, create kv cache, warmup model) took 34.59 seconds
2989 2989


Adding requests:   0%|          | 0/7473 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/29892 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

generate all response time: 724.9685311317444------------
valid sample: 20478 ; not valid sample: 9414


Saving the dataset (0/1 shards):   0%|          | 0/20478 [00:00<?, ? examples/s]

In [8]:
#filtered_train_ds['text'][0]
# when need to load from disk
from datasets import load_from_disk
filtered_train_ds = load_from_disk(training_data_output_dir)
original_ds = load_from_disk("dataset/original_training_data.hf")
filtered_train_ds
    


Dataset({
    features: ['text', 'prompt_temp', 'gt'],
    num_rows: 20478
})

In [5]:
# filtered_train_ds['text'][0]

In [6]:
# original_ds['text'][0]

'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAssistant: <think> Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May. </think> <answer> 72 </answer>'

In [9]:
filtered_train_ds_new = dict()
filtered_train_ds_new['text'] = []
for line in filtered_train_ds['text']:

    if 'Assistant: <think>' in line:
        filtered_train_ds_new['text'].append(line)


filtered_train_ds_new = Dataset.from_dict(filtered_train_ds_new)
filtered_train_ds_combined = interleave_datasets(
    [filtered_train_ds_new, original_ds],
    probabilities=[0.5, 0.5],
)
len(filtered_train_ds_combined['text'])

14762

### training

In [10]:
from peft import PeftModel
import numpy

#model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(
    initial_sft_model_dir,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    ).to(device)


sft_config = SFTConfig(
    max_seq_length=1024,
    output_dir="./checkpoint/expert_iteration/sft_iter1",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=32,
    learning_rate=8e-6,  
    num_train_epochs=5,
    logging_steps=10,
    dataset_text_field="text",
    label_names=["labels"],
    warmup_ratio=0.1,
    report_to = "wandb",  
    bf16=True,   
   
    #pad_token_id=eos_id,
    #eos_token_id=eos_id,          # <— this is what TRL will use to stop
    # you can also set other generation defaults here if you like
)


# # LoRA 配置
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=64,  
    lora_alpha=64,
    lora_dropout=0.1,
   
    modules_to_save=["embed_tokens", "lm_head"],
    target_modules='all-linear'
    #target_modules = ["q_proj","v_proj"]
)

# # 将LoRA配置应用到模型
peft_model = get_peft_model(model, lora_config)

# # use following when continue training
# #peft_model = PeftModel.from_pretrained(model, "./checkpoint/sft/sft_lora_results3/checkpoint-500", is_trainable=True ) 




# # 训练参数配置 (not support anymore)
# training_args = TrainingArguments(
#     output_dir="./checkpoint/sft/sft_lora_results",
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=1e-4,  
#     num_train_epochs=3,
#     fp16=True,  
#     logging_steps=1
# )

# 使用Trainer API进行训练
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=filtered_train_ds_combined,
    data_collator=collator, 
    args=sft_config,
   
    
    #data_collator=torch.utils.data.DataCollatorWithPadding(tokenizer=tokenizer)
)

#with torch.serialization.safe_globals([numpy.core.multiarray._reconstruct]):


trainer.train(resume_from_checkpoint=False)  #set to false when not continue training


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Converting train dataset to ChatML:   0%|          | 0/14762 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/14762 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14762 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14762 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: haoranyu66 (udacity_jeff) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,0.538500
20,0.532600
30,0.546900
40,0.529500
50,0.496500
60,0.470200
70,0.426600
80,0.410100
90,0.375600
100,0.345300


### Evaluation

In [3]:
def sft_evaluation(vllm,  val_ds,  device: str, out_dir):
    start_time = time.time()
    #format_reward, answer_reward, answer = [], [], []
    # total_response_len = 0
    # total_response_len_correct = 0
    # total_response_len_incorrect = 0
    # total_sample = 0
    # total_correct_sample = 0
    # total_incorrect_sample = 0
    response_avg_entropy_lst = [] #(batch,)
    response_len_lst = [] #(batch,)
    correct_lst = []
    incorrect_lst = []
    #initialize llm for vllm
    #llm = init_vllm(hf_policy_dir, device, 233)
    #load_policy_into_vllm_instance(policy, tokenizer, llm)
    sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["</answer>"]
    )
    sampling_params.include_stop_str_in_output = True
    all_prompt_texts = val_ds['prompt_temp']
    all_answer_gt = val_ds['gt']
    # for batch in val_dataset:
    #     all_prompt_texts.extend(batch['prompt_texts'])
    #     all_answer_gt.extend(batch['answer_gt'] ) 
   
    all_model_outputs = vllm.generate(all_prompt_texts, sampling_params)
    start2_time = time.time()
    print(f'generate all response time: {start2_time - start_time}------------')

    # still need policy model for eval mode
    #policy.eval()
    os.makedirs(out_dir.rsplit('/', 1)[0], exist_ok=True) 
    with open(out_dir, "a", encoding="utf-8") as f:
        format_reward, answer_reward, reward = 0, 0, 0
     
        for i, (output, gt) in enumerate(zip(all_model_outputs, all_answer_gt)):
            prompt = output.prompt
            generated_text = output.outputs[0].text
            
            res = r1_zero_reward_fn(generated_text, gt)
            # format_reward+= res['format_reward']
            # answer_reward+= res['answer_reward']
            # reward+= res['reward']
            #print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}, format_reward: {str(res['format_reward'])}, answer_reward: {str(res['answer_reward'])}, reward: {str(res['reward'])}")
            #final_output.append([res['format_reward'], res['answer_reward'], res['reward']])
            dp = {
                "prompt": f"{prompt}",
                "ground_truth": gt, 
                "output": f"{generated_text}",
                "format_reward": res['format_reward'],
                "answer_reward": res['answer_reward'],
                "reward": res['reward'],
                #"avg_response_token_entropy": response_avg_entropy_lst[i]
            }
            correct_lst.append(int(res['reward']==1))
            incorrect_lst.append(int(res['reward']!=1))
            format_reward+= res['format_reward']
            answer_reward+= res['answer_reward']
            reward+= res['reward']

            json.dump(dp, f, ensure_ascii=False)
            f.write("\n") 
            
    #total_response_len_correct = np.sum(np.array(correct_lst) * np.array(response_len_lst))
    #total_response_len_incorrect = np.sum(np.array(incorrect_lst) * np.array(response_len_lst))
    total_correct_sample = np.sum(np.array(correct_lst))
    total_incorrect_sample = np.sum(np.array(incorrect_lst))
    total_sample = len(all_model_outputs)
    #total_response_len = np.sum(np.array(response_len_lst))

    print(f'total_correct_sample: {total_correct_sample}, total_incorrect_sample: {total_incorrect_sample}')
    print(format_reward, answer_reward, reward)
    #print(f'final stats\navg_response_len={total_response_len/total_sample:.2f}, avg_response_len_correct={total_response_len_correct/total_correct_sample:.2f}, avg_response_len_incorrect={total_response_len_incorrect/total_incorrect_sample:.2f}')
    #print("some check:", total_response_len_correct+total_response_len_incorrect==total_response_len)
    #print("moer check:", total_correct_sample+total_incorrect_sample==total_sample)
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    return total_correct_sample / total_sample

In [4]:
model_name = "Qwen/Qwen2.5-1.5B"
tokenizer_dir =  "checkpoint/sft/tokenizer"
# orig_model_dir =      "checkpoint/sft/sft_lora_results3/checkpoint-935/adapter_model.safetensors"
model_dir = "checkpoint/expert_iteration/sft_iter1/checkpoint-1000"
#model_dir = "checkpoint/sft/sft_lora_results3/checkpoint-500"
eval_out_dir = 'eval/expert_iteration/sft_iter1/sft_eval.jsonl'
tokenizer =  AutoTokenizer.from_pretrained(tokenizer_dir, local_files_only=True)
# Load base model
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_dir)
print(len(tokenizer))
base_model.resize_token_embeddings(len(tokenizer))

# Load LoRA adapter
peft_model = PeftModel.from_pretrained(base_model, model_dir)

# Merge LoRA weights into base model
merged_model = peft_model.merge_and_unload()

# Save merged model and tokenizer in a new directory
hf_model_dir = "./checkpoint/expert_iteration/sft_iter1/sft_iter1_merged_model"
merged_model.save_pretrained(hf_model_dir)

tokenizer.save_pretrained(hf_model_dir)
del base_model
del peft_model
torch.cuda.empty_cache()
gc.collect()
vllm = init_vllm(hf_model_dir, device, 233)
sft_evaluation(vllm, val_ds , device, eval_out_dir)


151666
INFO 06-26 08:17:31 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-26 08:17:31 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 06-26 08:17:31 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-26 08:17:40 [config.py:793] This model supports multiple tasks: {'reward', 'embed', 'classify', 'score', 'generate'}. Defaulting to 'generate'.
INFO 06-26 08:17:40 [config.py:2118] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-26 08:17:46 [__init__.py:243] Automatically detected platform cuda.
INFO 06-26 08:17:49 [core.py:438] Waiting for init message from front-end.
INFO 06-26 08:17:49 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-26 08:17:49 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolve

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.10it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.10it/s]



INFO 06-26 08:17:50 [default_loader.py:280] Loading weights took 0.59 seconds
INFO 06-26 08:17:51 [gpu_model_runner.py:1549] Model loading took 2.9094 GiB and 0.780058 seconds
INFO 06-26 08:17:57 [backends.py:459] Using cache directory: /home/sagemaker-user/.cache/vllm/torch_compile_cache/9498212cc2/rank_0_0 for vLLM's torch.compile
INFO 06-26 08:17:57 [backends.py:469] Dynamo bytecode transform time: 6.85 s
INFO 06-26 08:18:03 [backends.py:132] Directly load the compiled graph(s) for shape None from the cache, took 4.958 s
INFO 06-26 08:18:04 [monitor.py:33] torch.compile takes 6.85 s in total
INFO 06-26 08:18:05 [kv_cache_utils.py:637] GPU KV cache size: 511,744 tokens
INFO 06-26 08:18:05 [kv_cache_utils.py:640] Maximum concurrency for 131,072 tokens per request: 3.90x
INFO 06-26 08:18:25 [gpu_model_runner.py:1933] Graph capturing finished in 20 secs, took 0.45 GiB
INFO 06-26 08:18:25 [core.py:167] init engine (profile, create kv cache, warmup model) took 34.47 seconds


Adding requests:   0%|          | 0/659 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/659 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

generate all response time: 23.31014394760132------------
total_correct_sample: 221, total_incorrect_sample: 438
538.0 221.0 221.0


np.float64(0.3353566009104704)

### load test result

In [7]:
import pandas as pd    
pd.set_option('display.max_colwidth', None)
eval_res = pd.read_json(path_or_buf=eval_out_dir, lines=True)
eval_res.head(200)
# with open(eval_out_dir) as f:
#     data = [json.loads(line) for line in f]
#with open(eval_out_dir, "r", encoding="utf-8") as f:

,prompt,ground_truth,output,format_reward,answer_reward,reward
0,"A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nAssistant: <think>",18,"Janet's ducks produce 16 eggs per day. She eats 3 eggs as breakfast daily. So, she ends up with 16 - 3 = 13 eggs. But she also need 4 eggs for muffins, that leaves 13 - 4 = 9 eggs. Each egg is sold for 2 dollars. Therefore, Janet will make 9 * 2 = 18 dollars with her sold eggs every day.</think>\n<answer>18</answer>",0,0,0
1,"A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?\nAssistant: <think>",3,"First, let's determine the amount of white fiber required for the robe. We are given that it takes 2 bolts of blue fiber and half that much white fiber. So, we have:</think> 2 bolts x (1/2) = 1 bolt of white fiber.\n\nNow, we know that the robe requires 2 bolts of blue fiber and 1 bolt of white fiber, making a total of:</think> 2 bolts + 1 bolt = 3 bolts.\n\nSo, the total number of bolts of fiber required to create the robe is 3.<answer> 3 </answer>",0,0,0
2,"A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Josh decides to try flipping a house. He buys a house for $80,000 and then puts in $50,000 in repairs. This increased the value of the house by 150%. How much profit did he make?\nAssistant: <think>",70000,"\nHis house is now worth $80,000 + 150% of $80,000 = $80,000 + $120,000 = $200,000\nHe financed this purchase with a loan of $280,000\nThe profit is $200,000 - $280,000 = -$80,000 </think>\n<answer> -$80,000 </answer>",0,0,0
3,"A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?\nAssistant: <think>",540,"He runs 3 set of sprints, so we multiply 3 by 60 to get distance for 3s of sprints. So the distance he ran is 180 meters of sprints. </think> <answer>180 meters</answer>",1,0,0
4,"A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </th

In [16]:
curr = eval_res.loc[1]['output']
res = r1_zero_reward_fn(curr, 3)
curr

" First, let's determine the amount of white fiber required for the robe. We are given that it takes 2 bolts of blue fiber and half that much white fiber. So, we have:</think> 2 bolts x (1/2) = 1 bolt of white fiber.\n\nNow, we know that the robe requires 2 bolts of blue fiber and 1 bolt of white fiber, making a total of:</think> 2 bolts + 1 bolt = 3 bolts.\n\nSo, the total number of bolts of fiber required to create the robe is 3.<answer> 3 </answer>"

In [10]:
filtered_train_ds["text"]

['A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAssistant: <think> Natalia sold clips to 48 of her friends in April <br> In May Natalia only sold half as much clips which is 48/2 = 24 clips <br> Thus, Natalia had 48 clips + 24 clips = 72 clips in April and May </think> <answer> 72 </answer>',
 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and the